# Build metadata CSV for `Dataset/raw_new_balanced`

This notebook creates **one file**:

- `Dataset/raw_new_balanced_metadata.csv`

It scans this folder structure:

- `Dataset/raw_new_balanced/ALSwDysarthria/<subjectID>/*.wav`
- `Dataset/raw_new_balanced/ALSwoDysarthria/<subjectID>/*.wav`
- `Dataset/raw_new_balanced/Control/<subjectID>/*.wav`

It **ignores** files created by the acoustic extractor:
- anything inside a `voiced/` directory
- any wav ending with `_OnlyVoiced.wav`

Important note about **subject IDs** in this dataset:
- Many `subjectID` folder names repeat across labels (e.g., `F1` exists under multiple labels).
- The notebook writes both:
  - `subjectID` = raw subject folder name
  - `subject_uid` = leakage-safe unique ID: `label__subjectID`

The output CSV contains:
- `label`, `subjectID`, `subject_uid`, `file_path`, `file_name`, `utterance_type`, `utterance_id`

`file_path` is written **relative to the workspace root** so the repo scripts can resolve it consistently.

In [1]:
from __future__ import annotations

import os
import re
from dataclasses import asdict, dataclass
from pathlib import Path

import pandas as pd

# ----------------------------
# Workspace + data locations
# ----------------------------

# Set this if auto-detection ever picks the wrong folder.
# Example: MANUAL_WORKSPACE_ROOT = "/mnt/d/PhD/ALS_Diagnosis_Meta"
MANUAL_WORKSPACE_ROOT = ""

# Notebook/kernel working directory (can vary depending on how you launch VS Code / Jupyter).
NOTEBOOK_CWD = Path.cwd()
print("Notebook CWD:", NOTEBOOK_CWD)

if MANUAL_WORKSPACE_ROOT:
    WORKSPACE_ROOT = Path(MANUAL_WORKSPACE_ROOT).expanduser()
    if not WORKSPACE_ROOT.is_dir():
        raise RuntimeError(f"MANUAL_WORKSPACE_ROOT does not exist: {WORKSPACE_ROOT}")
else:
    # Auto-detect workspace root by searching upward for a 'Dataset' folder.
    # This is safe as long as your project contains 'Dataset/' at the top level.
    WORKSPACE_ROOT = None
    for p in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]:
        if (p / "Dataset").is_dir():
            WORKSPACE_ROOT = p
            break
    if WORKSPACE_ROOT is None:
        raise RuntimeError("Could not find workspace root (a parent folder containing 'Dataset').")

WORKSPACE_ROOT = WORKSPACE_ROOT.resolve()
print("Workspace root:", WORKSPACE_ROOT)

DATA_ROOT = WORKSPACE_ROOT / "Dataset" / "raw_new_balanced"
OUT_CSV = WORKSPACE_ROOT / "Dataset" / "raw_new_balanced_metadata.csv"

print("Data root:", DATA_ROOT)
print("Output CSV:", OUT_CSV)

# Expected labels (folder names) under Dataset/raw_new_balanced
LABELS = ["ALSwDysarthria", "ALSwoDysarthria", "Control"]

if not DATA_ROOT.is_dir():
    raise RuntimeError(f"Data root not found: {DATA_ROOT}")

missing = [lbl for lbl in LABELS if not (DATA_ROOT / lbl).is_dir()]
if missing:
    raise RuntimeError(f"Missing label folders under {DATA_ROOT}: {missing}")

Notebook CWD: /mnt/d/PhD/ALS_Diagnosis_Meta/SOTA_models/Speech-Disorder-Classification-ML/src/feature_extraction
Workspace root: /mnt/d/PhD/ALS_Diagnosis_Meta
Data root: /mnt/d/PhD/ALS_Diagnosis_Meta/Dataset/raw_new_balanced
Output CSV: /mnt/d/PhD/ALS_Diagnosis_Meta/Dataset/raw_new_balanced_metadata.csv


In [4]:
_SUFFIX_RE = re.compile(r"_(?P<kind>[COW])(?P<id>\d+)$", re.IGNORECASE)

# By default we only scan wavs directly under each subject folder:
#   Dataset/raw_new_balanced/<label>/<subjectID>/*.wav
# This prevents accidental inclusion of derived/intermediate wavs in nested folders.
ALLOW_NESTED_WAVS = False

@dataclass(frozen=True)
class Row:
    label: str
    subjectID: str  # raw subject folder name
    subject_uid: str  # leakage-safe unique subject ID: label__subjectID
    file_path: str
    file_name: str
    utterance_type: str
    utterance_id: str


def parse_suffix(file_stem: str) -> tuple[str, str]:
    m = _SUFFIX_RE.search(file_stem)
    if not m:
        return "", ""
    return m.group("kind").upper(), m.group("id")


def should_skip_wav(path: Path) -> bool:
    # Skip extractor-generated voiced copies
    if any(part.lower() == "voiced" for part in path.parts):
        return True
    if path.stem.endswith("_OnlyVoiced"):
        return True
    return False


def iter_wavs(data_root: Path):
    for label in LABELS:
        class_dir = data_root / label
        for subject_dir in sorted([p for p in class_dir.iterdir() if p.is_dir()]):
            subject_id = subject_dir.name
            wav_iter = subject_dir.rglob("*.wav") if ALLOW_NESTED_WAVS else subject_dir.glob("*.wav")
            for wav_path in sorted(wav_iter):
                if not wav_path.is_file():
                    continue
                if should_skip_wav(wav_path):
                    continue
                # Extra safety: ensure the file is actually under the expected data_root
                try:
                    wav_path.resolve().relative_to(data_root.resolve())
                except Exception:
                    continue
                yield label, subject_id, wav_path


def build_metadata(progress_every: int = 1000) -> pd.DataFrame:
    rows: list[Row] = []

    ws_abs = os.path.abspath(str(WORKSPACE_ROOT))

    for i, (label, subject_id, wav_path) in enumerate(iter_wavs(DATA_ROOT), start=1):
        wav_abs = os.path.abspath(str(wav_path))
        rel_path = os.path.relpath(wav_abs, start=ws_abs)
        # Guard against any accidental paths that escape the workspace (shouldn't happen).
        if rel_path.startswith(".." + os.sep) or rel_path == "..":
            raise RuntimeError(f"Found wav outside WORKSPACE_ROOT: {wav_path} (rel={rel_path})")

        utt_type, utt_id = parse_suffix(wav_path.stem)
        subject_uid = f"{label}__{subject_id}"

        rows.append(
            Row(
                label=label,
                subjectID=subject_id,
                subject_uid=subject_uid,
                file_path=rel_path,
                file_name=wav_path.name,
                utterance_type=utt_type,
                utterance_id=utt_id,
            )
        )

        if progress_every and i % progress_every == 0:
            print(f"Scanned {i} wavs...")

    df = pd.DataFrame([asdict(r) for r in rows])
    df = df.sort_values(["label", "subjectID", "file_name"], kind="mergesort").reset_index(drop=True)
    return df


df_meta = build_metadata(progress_every=1000)
df_meta.head()

Scanned 1000 wavs...
Scanned 2000 wavs...
Scanned 3000 wavs...
Scanned 4000 wavs...
Scanned 5000 wavs...
Scanned 6000 wavs...


,label,subjectID,subject_uid,file_path,file_name,utterance_type,utterance_id
0,ALSwDysarthria,F1,ALSwDysarthria__F1,Dataset/raw_new_balanced/ALSwDysarthria/F1/F1_...,F1_ALS_F79_KAJ_49847901_20191223_C1.wav,C,1
1,ALSwDysarthria,F1,ALSwDysarthria__F1,Dataset/raw_new_balanced/ALSwDysarthria/F1/F1_...,F1_ALS_F79_KAJ_49847901_20191223_C10.wav,C,10
2,ALSwDysarthria,F1,ALSwDysarthria__F1,Dataset/raw_new_balanced/ALSwDysarthria/F1/F1_...,F1_ALS_F79_KAJ_49847901_20191223_C11.wav,C,11
3,ALSwDysarthria,F1,ALSwDysarthria__F1,Dataset/raw_new_balanced/ALSwDysarthria/F1/F1_...,F1_ALS_F79_KAJ_49847901_20191223_C12.wav,C,12
4,ALSwDysarthria,F1,ALSwDysarthria__F1,Dataset/raw_new_balanced/ALSwDysarthria/F1/F1_...,F1_ALS_F79_KAJ_49847901_20191223_C13.wav,C,13


In [5]:
print("Total wavs:", len(df_meta))
print("Counts by label:")
print(df_meta["label"].value_counts().to_string())

print("\nCounts by utterance_type (only parsed suffixes):")
vc_kind = df_meta.loc[df_meta["utterance_type"].astype(str) != "", "utterance_type"].value_counts()
print(vc_kind.to_string() if len(vc_kind) else "(none)")

print("\nUnique subject folders overall (raw subject folder names):", df_meta["subjectID"].nunique())
print("Unique subjects per label (raw subject folder names):")
print(df_meta.groupby("label")["subjectID"].nunique().to_string())

print("\nUnique subjects overall (leakage-safe subject_uid = label__subjectID):", df_meta["subject_uid"].nunique())
print("Unique subjects per label (subject_uid):")
print(df_meta.groupby("label")["subject_uid"].nunique().to_string())

# Detect subjectID collisions across labels
labels_per_subject = df_meta.groupby("subjectID")["label"].nunique().sort_values(ascending=False)
collisions = labels_per_subject[labels_per_subject > 1]
print("\nRaw subject folder names appearing in multiple labels:", int(collisions.shape[0]))
if len(collisions):
    print("Example collisions (first 10):")
    print(collisions.head(10).to_string())

Total wavs: 6200
Counts by label:
label
ALSwDysarthria     2600
Control            2000
ALSwoDysarthria    1600

Counts by utterance_type (only parsed suffixes):
utterance_type
W    3162
C    2108
O     930

Unique subject folders overall (raw subject folder names): 29
Unique subjects per label (raw subject folder names):
label
ALSwDysarthria     26
ALSwoDysarthria    16
Control            20

Unique subjects overall (leakage-safe subject_uid = label__subjectID): 62
Unique subjects per label (subject_uid):
label
ALSwDysarthria     26
ALSwoDysarthria    16
Control            20

Raw subject folder names appearing in multiple labels: 21
Example collisions (first 10):
subjectID
F1    3
F2    3
M7    3
M9    3
M8    3
M2    3
M3    3
M4    3
M5    3
M6    3


In [6]:
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_meta.to_csv(OUT_CSV, index=False)
print("Wrote:", OUT_CSV)

# Show first few rows as a sanity check
pd.read_csv(OUT_CSV).head()

Wrote: /mnt/d/PhD/ALS_Diagnosis_Meta/Dataset/raw_new_balanced_metadata.csv


,label,subjectID,subject_uid,file_path,file_name,utterance_type,utterance_id
0,ALSwDysarthria,F1,ALSwDysarthria__F1,Dataset/raw_new_balanced/ALSwDysarthria/F1/F1_...,F1_ALS_F79_KAJ_49847901_20191223_C1.wav,C,1
1,ALSwDysarthria,F1,ALSwDysarthria__F1,Dataset/raw_new_balanced/ALSwDysarthria/F1/F1_...,F1_ALS_F79_KAJ_49847901_20191223_C10.wav,C,10
2,ALSwDysarthria,F1,ALSwDysarthria__F1,Dataset/raw_new_balanced/ALSwDysarthria/F1/F1_...,F1_ALS_F79_KAJ_49847901_20191223_C11.wav,C,11
3,ALSwDysarthria,F1,ALSwDysarthria__F1,Dataset/raw_new_balanced/ALSwDysarthria/F1/F1_...,F1_ALS_F79_KAJ_49847901_20191223_C12.wav,C,12
4,ALSwDysarthria,F1,ALSwDysarthria__F1,Dataset/raw_new_balanced/ALSwDysarthria/F1/F1_...,F1_ALS_F79_KAJ_49847901_20191223_C13.wav,C,13


In [7]:
# Validate paths exist (fast sample check + optional full check)

sample_n = 200
sample = df_meta["file_path"].astype(str).head(sample_n)
exists_sample = [(WORKSPACE_ROOT / p).exists() for p in sample]
print(f"Exists check (first {sample_n}): {sum(exists_sample)}/{sample_n}")

RUN_FULL_EXISTS_CHECK = False  # set True if you want to check all 6200
if RUN_FULL_EXISTS_CHECK:
    all_paths = df_meta["file_path"].astype(str)
    ok = sum((WORKSPACE_ROOT / p).exists() for p in all_paths)
    print(f"Exists check (all): {ok}/{len(df_meta)}")

Exists check (first 200): 200/200
